In [1]:
import pandas as pd

ratings = pd.read_csv(
    "../data/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

items = pd.read_csv(
    "../data/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=["item_id", "title"] + [f"col_{i}" for i in range(22)],
    usecols=[0, 1]
)

print(ratings.shape, items.shape)
print(items.head())

(100000, 4) (1682, 2)
   item_id              title
0        1   Toy Story (1995)
1        2   GoldenEye (1995)
2        3  Four Rooms (1995)
3        4  Get Shorty (1995)
4        5     Copycat (1995)


In [2]:
import sys
sys.path.append("../src")

from recommender import build_user_item_matrix, train_svd_model, recommend_top_n

user_item_matrix = build_user_item_matrix(ratings)
print("Matrix shape:", user_item_matrix.shape)
print(user_item_matrix.iloc[:5, :5])

Matrix shape: (943, 1682)
item_id    1    2    3    4    5
user_id                         
1        5.0  3.0  4.0  3.0  3.0
2        4.0  0.0  0.0  0.0  0.0
3        0.0  0.0  0.0  0.0  0.0
4        0.0  0.0  0.0  0.0  0.0
5        4.0  3.0  0.0  0.0  0.0


In [3]:
reconstructed_df, svd_model = train_svd_model(user_item_matrix, n_components=20)
print("Reconstructed matrix shape:", reconstructed_df.shape)
print("Explained variance ratio (sum):", svd_model.explained_variance_ratio_.sum())

Reconstructed matrix shape: (943, 1682)
Explained variance ratio (sum): 0.41213944734463087


In [4]:
sample_user = 1
recommendations = recommend_top_n(sample_user, reconstructed_df, user_item_matrix, items, n=10)
print(f"Top 10 recommendations for user {sample_user}:")
print(recommendations[["title", "predicted_score"]])

Top 10 recommendations for user 1:
                                                 title  predicted_score
474                               Trainspotting (1996)         3.958260
422                  E.T. the Extra-Terrestrial (1982)         3.361876
482                                  Casablanca (1942)         3.263087
274                       Sense and Sensibility (1995)         3.241236
432                                    Heathers (1989)         3.226547
407                              Close Shave, A (1995)         3.159864
654                                 Stand by Me (1986)         3.128734
317                            Schindler's List (1993)         3.114066
402                                      Batman (1989)         3.072493
473  Dr. Strangelove or: How I Learned to Stop Worr...         2.949152


In [5]:
from recommender import recommend_for_new_user

new_user_likes = [1, 50]  # e.g. this new user said they liked Toy Story (1) and Star Wars (50)
cold_start_recs = recommend_for_new_user(new_user_likes, ratings, items, n=10)
print("Recommendations for a brand-new user who liked Toy Story and Star Wars:")
print(cold_start_recs["title"])

Recommendations for a brand-new user who liked Toy Story and Star Wars:
11                             Usual Suspects, The (1995)
63                       Shawshank Redemption, The (1994)
113     Wallace & Gromit: The Best of Aardman Animatio...
168                            Wrong Trousers, The (1993)
177                                   12 Angry Men (1957)
317                               Schindler's List (1993)
407                                 Close Shave, A (1995)
482                                     Casablanca (1942)
602                                    Rear Window (1954)
1448                               Pather Panchali (1955)
Name: title, dtype: str


## Recommender System Summary

- **Existing users:** Matrix Factorization (Truncated SVD, 20 components, 41.2% explained variance) generates personalized Top-N recommendations from the user-item ratings matrix.
- **Cold start (new users):** Falls back to popularity-weighted recommendations among users who share at least one known preference, filtered to items with 5+ ratings for reliability.
- Example: a new user who liked Toy Story and Star Wars received Shawshank Redemption, Schindler's List, and Casablanca — consistent, well-regarded picks rather than random guesses.